In [ ]:
import AA500
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
def fit_linear_function(linear_range, x, y):
    coeff = np.polyfit(linear_range[x], linear_range[y], 1)
    r2 = np.corrcoef(linear_range[x], linear_range[y])[0,1]**2
    return coeff, r2

def plot_linear_fit(data, linear_range, x, y, text_start_x, text_start_y, text_y_spacing, ax):
    coeff, r2 = fit_linear_function(linear_range, x, y)
    vals= np.polyval(coeff, data[x])
    ax.plot(data[x], vals, color='b')
    ax.text(text_start_x, text_start_y, 'Slope: %.4f' % (coeff[0]))
    ax.text(text_start_x, text_start_y - text_y_spacing, 'Intercept: %.4f' % (coeff[1]))
    ax.text(text_start_x, text_start_y - 2*text_y_spacing, 'R2: %.4f' % (r2))
    return coeff, r2

# 5-21-25 Nutrient Analyzer Run QAQC

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/20250331 Run/'

result_05212025 = AA500.AA500_Result(result_dir + 'RCEW_NOx-PO4-NH4_Apr2025_2_2.csv', result_dir+'samplelist_20250331_2.xlsx')

## Drift and Baseline

In [ ]:
result_05212025.result_df

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,11), nrows = 3)

result_05212025.plot_QA('Nitrate', ax=ax[0], title='Nitrate')
result_05212025.plot_QA('Phosphate', ax=ax[1], title='Phosphate')
result_05212025.plot_QA('Ammonium', ax=ax[2], title='Ammonium')
fig.tight_layout()

- drift and baseline all look good except last ammonium drift

## Blank Check

In [ ]:
result_05212025.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- interesting, one DI Blank was pretty bad. contamination or off by one sample?

In [ ]:
result_05212025.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

In [ ]:
result_05212025.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- reagent blanks look good for nitrate and phosphate. 
- ammonium mean is high, but is pretty similar to DI blanks, so this seems like a problem with ammonium standards or milliQ is contaminated

## AA500 In-Sample Variation

In [ ]:
result_05212025.result_df.loc['RME'][['Nitrate std', 'Phosphate std', 'Ammonium std']].describe()

nitrate:
- mean of 1.7 ug/L great- SEAl reported 3-5 ug/L for this range
- most samples ok, but a few bad ones - throw those out
- qa threshold of 5 ug/l

phosphate:
- also great- mean of 3 ug/L in range of 4 ug/L in SEAL datasheet
- also a bad one or two
- qa threshold of 4 ug/l

ammonium:
- way better than seal spec! 75% less than 1 ug/L, seal spec is 2 ug/L
- one bad one with a max of 7 ug/L
- qa threshold of 2 ug/l


In [ ]:
result_05212025.result_df.loc['RME']

In [ ]:
result_05212025.result_df.loc['Dobson']

- nice. for nitrate QA flag, RME has only one, Dobson has zero. 
- most flags for phosphate, a couple for ammonium. but no wholesale bad data

## Between Bottle Variation

In [ ]:
def plot_between_bottle_bar(result_df, analyte, **kwargs):
    # Reset index to ensure 'Sample Datetime' and 'Bottle Replicate' are columns
    df = result_df.reset_index()

    # Pivot so each bottle replicate is a column, indexed by date
    pivoted = df.pivot_table(
    index='Sample Datetime',
    columns='Bottle Replicate',
    values=analyte + ' mean'
    )

    # Optionally, get errors for yerr
    yerr = df.pivot_table(
        index='Sample Datetime',
        columns='Bottle Replicate',
        values=analyte + ' err'
    )

    # Plot
    ax = pivoted.plot(kind='bar', yerr=yerr, rot=45, **kwargs)
    ax.set_ylabel(analyte+ ' mean')
    #ax.set_title(analyte + ' by Date and Bottle Replicate')
    ax.legend(title='Bottle Replicate')
    #plt.tight_layout()
    #plt.show()
    return ax

plot_between_bottle_bar(result_05212025.result_df.loc['RME'], 'Nitrate')

#fig, ax=  plt.subplots()
#first_rep = result_05212025.result_df['Bottle Replicate']==1

#result_05212025.result_df.loc['RME'].groupby('Bottle Replicate').plot(y='Nitrate mean',ax=ax, yerr= 'Nitrate err',kind='bar', rot=45)


- looks pretty good! good matches betweenn bottles except the 24th
- AA500 variation looks pretty good too, only the 2/25 sample is pretty bad
- these are also samples that i filtered... 

In [ ]:
plot_between_bottle_bar(result_05212025.result_df.loc['Dobson'], 'Nitrate')

#fig, ax=  plt.subplots(result_05212025.loc['Dobson'])
#first_rep = result_05212025.result_df['Bottle Replicate']==1

#result_05212025.result_df[first_rep].loc['Dobson'].reset_index().plot(x='Sample Datetime', y='Nitrate mean',ax=ax, yerr= 'Nitrate err',kind='scatter', rot=45, label = 'First Bottle')
#result_05212025.result_df[~first_rep].loc['Dobson'].reset_index().plot(x='Sample Datetime', y='Nitrate mean',ax=ax, yerr= 'Nitrate err',kind='scatter', rot=45,color='r', label = 'Second Bottle', title = 'Dobson Nutrient Analyzer Data from 5-21-25 Run')

- yikes. Dobson data is straight trash. huge variability between bottle replicates.
- I suspect this has to less to do with it being Dobson or RME and more to do with it being samples I filtered vs samples josh filtered. Or could be that the RME samples were filtered very freshly.

In [ ]:
fig, ax = plt.subplots(nrows=3, figsize=(11,8.5), sharex=True)

plot_between_bottle_bar(result_05212025.result_df.loc['Dobson'], 'Nitrate', ax=ax[0], title = 'Nitrate BBV')
plot_between_bottle_bar(result_05212025.result_df.loc['Dobson'], 'Phosphate', ax=ax[1], title = 'Phosphate BBV')
plot_between_bottle_bar(result_05212025.result_df.loc['Dobson'], 'Ammonium', ax=ax[2], title = 'Ammonium BBV')


fig.tight_layout()




# 6-10-25 Nutrient Analyzer Run QAQC Run 1

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/20250610 Run/'

result_061025_1 = AA500.AA500_Result(result_dir + 'RCEW_NOx_PO4_NH3_11Jun25_R1.TXT', result_dir+'samplelist_20250610_1.xlsx')

result_061025_1.result_df

## Drift and Baseline

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,11), nrows = 3)

result_061025_1.plot_QA('Nitrate', ax=ax[0], title='Nitrate')
result_061025_1.plot_QA('Phosphate', ax=ax[1], title='Phosphate')
result_061025_1.plot_QA('Ammonium', ax=ax[2], title='Ammonium')
fig.tight_layout()

- nitrate ran out of reagent after 8th drift standard - need to rerun cups 91 on.
- ammonium drifted after 10th drift standard

## Blank Check

### Instrument Blank

In [ ]:
result_061025_1.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- all nitrate blanks look good. 
- phosphate and ammonium blanks comparable to 5-21-25, but lacking context in expected phosphate and ammonium levels

In [ ]:
result_061025_1.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

### Reagent blank

In [ ]:
result_061025_1.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

In [ ]:
result_061025_1.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

- no nitrate contamination from reagents
- third reagent blank has some phosphate and ammonium contamination from the sodium carbonate it looks like - could be from tip into sample

### Field Blank


In [ ]:
result_061025_1.result_df.loc['Field Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

In [ ]:
result_061025_1.result_df.loc['Field Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

- no contamination from field blanks

## AA500 In-Sample Variation

In [ ]:
result_061025_1.result_df.loc['RME'][['Nitrate std', 'Phosphate std', 'Ammonium std']].describe()

nitrate:
- mean of .9 ug/L great- SEAl reported 3-5 ug/L for this range
- most samples ok, but a few bad ones - throw those out
- qa threshold of 5 ug/l

phosphate:
- ok - mean of 6 ug/L greater than 4 ug/L in SEAL datasheet
- also a bad one or two - max of 38 ug/L!
- qa threshold of 4 ug/l

ammonium:
- way better than seal spec! 75% less than 1.5 ug/L, seal spec is 2 ug/L
- one bad one with a max of 8 ug/L
- qa threshold of 2 ug/l


In [ ]:
result_061025_1.result_df.loc['RME']

In [ ]:
result_061025_1.result_df.loc['Dobson']

- nice. for nitrate QA flag, RME has only a few
- most flags for phosphate, a couple for ammonium. but no wholesale bad data
- several really high phosphate readings (~.2 mg/L) that get flagged with a relatively low std (<8 ug/L). could up the threshold

## Between Bottle Variation

In [ ]:
plot_between_bottle_bar(result_061025_1.result_df.loc['RME'], 'Nitrate')


In [ ]:
fig, ax = plt.subplots(nrows=3, figsize=(11,8.5), sharex=True)

plot_between_bottle_bar(result_061025_1.result_df.loc['Dobson'], 'Nitrate', ax=ax[0], title = 'Nitrate BBV')
plot_between_bottle_bar(result_061025_1.result_df.loc['Dobson'], 'Phosphate', ax=ax[1], title = 'Phosphate BBV')
plot_between_bottle_bar(result_061025_1.result_df.loc['Dobson'], 'Ammonium', ax=ax[2], title = 'Ammonium BBV')


fig.tight_layout()




- not perfect, but better than than 5-21 run. are these my or josh's filtrates that are good?

### Is who filtered the samples responsible for the variation?

In [ ]:
ax= plot_between_bottle_bar(result_061025_1.result_df[result_061025_1.result_df['Filtered By']=='JCS'], 'Nitrate', title= 'Josh')


In [ ]:
ax = plot_between_bottle_bar(result_061025_1.result_df[result_061025_1.result_df['Filtered By']=='KF'], 'Nitrate', title= 'Kyle')

In [ ]:
ax = plot_between_bottle_bar(result_061025_1.result_df[result_061025_1.result_df['Filtered By']=='BE'], 'Nitrate', title= 'Ben')

- yes, Josh's show more variation. kyle has a bad one, and mine aren't perfect though

In [ ]:
result_061025_1.result_df.loc['RME']

In [ ]:
result_061025_1.result_df.loc['Dobson']

- RME has some BBV and ISV flags, but not many. Dobson has far more. most RME data look usable
- remember to only plot data from sites that are of type Hand, Autosampler, or Manual

# 6-10-25 Nutrient Analyzer Run 2 QAQC
## Drift and Baseline

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/20250610 Run/'

result_061025_2 = AA500.AA500_Result(result_dir + 'RCEW_NOx_PO4_NH3_11Jun25_R2.TXT', result_dir+'samplelist_20250610_2.xlsx')

result_061025_2.result_df

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,11), nrows = 3)

result_061025_2.plot_QA('Nitrate', ax=ax[0], title='Nitrate')
result_061025_2.plot_QA('Phosphate', ax=ax[1], title='Phosphate')
result_061025_2.plot_QA('Ammonium', ax=ax[2], title='Ammonium')
fig.tight_layout()

- no changes in drift or baseline

## Blank Check

### Instrument Blank

In [ ]:
result_061025_2.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- nitrate blank is a little high but not super out ouf raneg
- phoshpate and ammonium blank similar to previous runs

### Reagent blank

In [ ]:
result_061025_2.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- nitrate blank a littel high and similar to Di blank - suggests contamination of milliQ water
- phosphate and ammonium are similar to instrumetn blank, suggesting no contamination from reagent

## AA500 In-Sample Variation

In [ ]:
result_061025_2.result_df.loc['RME'][['Nitrate std', 'Phosphate std', 'Ammonium std']].describe()

nitrate:
- mean of 1.74 ug/L great- SEAl reported 3-5 ug/L for this range
- most samples are great, but one real bad one - 35 ug/L
- qa threshold of 5 ug/l

phosphate:
- great! - mean of 2 ug/L less than 4 ug/L in SEAL datasheet
- looks like only one kinda bad one at 9 ug/L
- qa threshold of 4 ug/l

ammonium:
- ammonium good - mean of 1 ug/L
- one bad one with a max of 14 ug/L
- qa threshold of 2 ug/l


In [ ]:
result_061025_2.result_df.loc['RME']

- only two rows with no flags

## Between Bottle Variation

In [ ]:
fig, ax = plt.subplots(nrows=3, figsize=(11,8.5), sharex=True)

plot_between_bottle_bar(result_061025_2.result_df.loc['RME'], 'Nitrate', ax=ax[0], title = 'Nitrate BBV')
plot_between_bottle_bar(result_061025_2.result_df.loc['RME'], 'Phosphate', ax=ax[1], title = 'Phosphate BBV')
plot_between_bottle_bar(result_061025_2.result_df.loc['RME'], 'Ammonium', ax=ax[2], title = 'Ammonium BBV')


fig.tight_layout()




- interesting. phosphate BBV looks better than nitrate. nitrate is actually kinda bad, lots of BBV flags and variability between samples. 
- inconsistent pattern between variation in different analytes. suggests that the problem isn't DI dilution, but contamination from individual analytes.
- could be contamination from bottle or sample cup? 
    - blanks would probably be contaminated too if it was a sample cup issue
    - so likely to be bottle? 
    - could also be contamination from sample thawing water - rerun?

# 8-11-25 Nutrient Analyzer Run

## Drift 

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/20250718 Run/'

result_071825 = AA500.AA500_Result(result_dir + 'RCEW_NOx_PO4_NH3_18Jul25.csv', result_dir+'samplelist_20250718.xlsx')

result_071825.result_df

In [ ]:
fig, ax = plt.subplots(figsize=(8.5,11), nrows = 3)

result_071825.plot_QA('Nitrate', ax=ax[0], title='Nitrate')
result_071825.plot_QA('Phosphate', ax=ax[1], title='Phosphate')
result_071825.plot_QA('Ammonium', ax=ax[2], title='Ammonium')
fig.tight_layout()

- nitrate and phosphate drift and baseline look good
slight change in ammonium drift standrads

## Blanks

### Instrument Blanks

In [ ]:
result_071825.result_df.loc['DI Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- excellent nitrate blanks!
- phosphate and ammonium blanks cmparable to previous runs
- ammoniuum max of 15 ug/L lower than instrument detection limit, so seems fine.

### Reagent Blanks

In [ ]:
result_071825.result_df.loc['Reagent Blank'][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

- no contamination of milliQ blanks from sodium carbonate
- slight phosphate contamination - ~5 ug/L
- most serious problem is contamination of ammonium - about 27 ug/L. Not serious enough to redo run, but keep in mind if you ever analyze ammonia data. how high is it normally?

### Bottle Blanks

In [ ]:
result_dir = '/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Standard Checks/'

stdcheck3_results = AA500.AA500_Result(result_dir + 'StdCheck_NOx-PO4-NH3_Aug1225_3.TXT', result_dir+'StdCheck_Aug1225_3_samplelist.xlsx')

stdcheck3_results.result_df

In [ ]:
std_desc = stdcheck3_results.result_df.groupby('Site Name')[['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()
std_desc_mean = std_desc.xs('mean', axis=1, level=1).loc[['Anna Bottle Blank']]
std_desc_std = std_desc.xs('std', axis=1, level=1).loc[['Anna Bottle Blank']]



In [ ]:
# Get the describe DataFrame
desc = result_071825.result_df.groupby('Site Name')[['Nitrate mean', 'Phosphate mean', 'Ammonium mean']].describe()

# Select the 'mean' row for each column
blankmeans = desc.xs('mean', axis=1, level=1).drop(['Dobson', 'Field Blank', 'RME', 'RME Snow'])
blankmeans = pd.concat([blankmeans, std_desc_mean])
blankstds = desc.xs('std', axis=1, level=1).drop(['Dobson', 'Field Blank', 'RME', 'RME Snow'])
blankstd = pd.concat([blankstds, std_desc_std])

fig, ax = plt.subplots(figsize=(11,8.5))

blankmeans.plot(kind='bar', ax=ax, yerr=blankstds)
#std_desc_mean.plot(kind='bar', ax=ax, yerr = std_desc_std)

- so all blanks look ok with exception of acid bath and NaOH HCl Blank. let's double check without them plotted for better scale:

In [ ]:
blankmeans.drop(['Acid Bath', 'NaOH HCl Blank']).plot(kind='bar', figsize=(11,8.5),yerr=blankstds)

- so blanks for nitrate look good, all about the same within error and low
- for phosphate, it looks like the wash bottle is contaminated with about 10 ug/L phosphate because reagent blank and DI blank (from wash bottle) have a bit of phosphate, while other blanks (from DI dispenser or carboy directly) don't
    - contamination is significant, LOD is 2 ug/L
- looks like everything is a bit contaminated with ammonium, but doesn't look that significant. except reagent blank. seems like one sample was really contaminated.
    - actually, LOD is 27 ug/L so ammonia blanks look good. if we went down to a .8 mg/L top standard we could get it down to 7 ug/L
    
- takeaways:
    - there's some ammonia contamination in the bottles, but data from bottles should definitely be usable for nitrate and phohpshate
    - rerun bottle blanks from newly washed bottles - are they really much better when left for a week?
    - you could rewash bottles you ahve if you're really concerned about ammonia, but they should be fine for nitrate.


In [ ]:
stdcheck3_results.result_df.loc[['NaOH HCl Blank', 'Acid Bath']][['Nitrate mean', 'Phosphate mean', 'Ammonium mean']]

- only one of each - maybe should do replicates. 
- NaOH HCl blank was astronoically high in nitrate, pretty normal in ammonia and phosphate
- Acid bath was high in nitrate and ammonia

- acid bath was definitely contaminated with ammonia because NaOH HCl Blank much lower concentration
- hard to say if acid bath was contaminated in nitrate - NaOH HCl also high, so could've been NaOH used to neutralize acid bath, or HCl used to neutralize NaOH sample... 
- doesn't really matter since we threw out the acid bath

Todo: 
- rerun new bottle blanks after they've sat for a week and rerun

## AA500 in-Sample Variation

In [ ]:
result_071825.result_df.loc['RME'][['Nitrate std', 'Phosphate std', 'Ammonium std']].describe()

nitrate:
- mean of3 ug/L great- SEAl reported 3-5 ug/L for this range
- most samples are great, but one real bad one - 23 ug/L
- qa threshold of 5 ug/l

phosphate:
- not very good - mean of 16 ug/L. 
- 25% of 3.5 ug/L, meaning ~75% of samples will be above QA threshold of 4 ug/L
- take a look at peaoss

ammonium:
- ammonium good - mean of 1.7 ug/L
- one bad one with a max of 7 ug/L
- qa threshold of 2 ug/l


## Between Bottle Variation

In [ ]:
fig, ax = plt.subplots(nrows=2, figsize=(11,8.5), sharex=True, sharey=True)

plot_between_bottle_bar(result_061025_2.result_df.loc['RME'], 'Nitrate', ax=ax[0], title = 'Nitrate BBV 061025')
plot_between_bottle_bar(result_071825.result_df.loc['RME'], 'Nitrate', ax=ax[1], title = 'Nitrate BBV 071825')


fig.tight_layout()




- more samples appear to closer to one another than 61025 run
- nitrate levels generally higher than 61025 run too

In [ ]:
fig, ax = plt.subplots(nrows=2, figsize=(11,8.5), sharex=True, sharey=True)

plot_between_bottle_bar(result_061025_2.result_df.loc['RME'], 'Phosphate', ax=ax[0], title = 'Phosphate BBV 061025')
plot_between_bottle_bar(result_071825.result_df.loc['RME'], 'Phosphate', ax=ax[1], title = 'Phosphate BBV 071825')


fig.tight_layout()

- in sample variability is shit for this run of phosphorous. why.
- phosphate levels look different, but not necessarily uniformly higher or lower than previous run.

In [ ]:
fig, ax = plt.subplots(nrows=2, figsize=(11,8.5), sharex=True, sharey=True)

plot_between_bottle_bar(result_061025_2.result_df.loc['RME'], 'Ammonium', ax=ax[0], title = 'Ammonium BBV 061025')
plot_between_bottle_bar(result_071825.result_df.loc['RME'], 'Ammonium', ax=ax[1], title = 'Ammonium BBV 071825')


fig.tight_layout()

# Joining All Nutrient Analyzer Runs

In [ ]:
class ReynoldsResult(pd.DataFrame):
    def get_all_params(self):

        return [col.split()[0] for col in self.columns if col.endswith(' QA')]
    
    def get_valid_data(self):
        df = ReynoldsResult(self.copy())
        params = df.get_all_params()
        for param in params:
            valid = (df[param + ' QA'] == '')
            df.loc[~valid, [param + ' mean', param + ' err', param + ' std']] = np.NaN
        return df

    def get_sample_data(self):
        df = ReynoldsResult(self.copy())
        sample = (df['Sample Type'] == 'Autosampler') | (df['Sample Type'] == 'Hand') | (df['Sample Type'] == 'Manual')
        return df.loc[sample]

In [ ]:
nutrient_results = ReynoldsResult(pd.concat([result_05212025.result_df, result_061025_1.result_df, result_071825.result_df]).sort_index())
nutrient_results

# Plotting Nutrient Analyzer Data with SCAN Data

## Timeseries

In [ ]:
dobson_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/Dobson/Processed Data/dobson_cleaned.csv', index_col='Date/Time', parse_dates=True)
dobson_cleaned

In [ ]:
rme_cleaned = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv', index_col='Date/Time', parse_dates=True)
rme_cleaned

In [ ]:
def plot_nutrient_data(df, start_date, end_date, site, param, ax, check_valid = True, **kwargs):
    first_rep = df['Bottle Replicate'] == 1

    df[first_rep].loc[site].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, label = 'First Bottle')
    df[~first_rep].loc[site].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', color = 'r', rot=45, label = 'Second Bottle')


def plot_nutrient_data_by_rundate(df, start_date, end_date, site, param, ax, check_valid=True, **kwargs):
    rundates = df['AA500 Run Date'].unique()
    from itertools import cycle
    color_cycler = plt.rcParams['axes.prop_cycle'].by_key()['color']
    colors = cycle(color_cycler)
    for date, color in zip(rundates, colors):
        df[df['AA500 Run Date'] == date].loc[site].loc[start_date:end_date].reset_index().plot(x='Sample Datetime', y=param+' mean',ax=ax, yerr= param+ ' err',kind='scatter', rot=45, color=color, label = 'Run on: ' + date.strftime('%m/%d/%Y'))
    

### RME

In [ ]:

fig, ax= plt.subplots(figsize=(11, 4.25))



rme_cleaned.loc['2-13-25':'05-01-25'].plot(y= ['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax, rot=45)
plot_nutrient_data(nutrient_results.get_sample_data().dropna(subset='Nitrate mean'), '2/13/25', '05/01/25', 'RME', 'Nitrate', ax)



- well cool. there's a couple outliers, on 2/15/25 3:30:00 and another 2-22-25 11:00. but overall theses data catch the general form of the S::CAN data. we can work with it.
- best fit looks to be two_wavelength_no3_mgl_correct, seems to fit better than uncorrected version, which is second best. 
- one_wavelength corrected not plotted, its way off

In [ ]:
fig, ax= plt.subplots(figsize=(11,8.5), nrows= 2)

rme_cleaned.loc['03-21-25':'04-01-25'].plot(y= ['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax[0], rot=45, title= 'By Bottle')
plot_nutrient_data(nutrient_results.get_sample_data().dropna(subset='Nitrate mean'), '3/21/25', '04/01/25', 'RME', 'Nitrate', ax[0])


rme_cleaned.loc['03-21-25':'04-01-25'].plot(y= ['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax[1], rot=45, title= "By AA500 Run")
plot_nutrient_data_by_rundate(nutrient_results.get_sample_data().dropna(subset='Nitrate mean'), '3/21/25', '04/01/25', 'RME', 'Nitrate', ax[1])


In [ ]:
good_results['AA500 Run Date'].unique()

In [ ]:
fig, ax= plt.subplots(figsize=(11,8.5))

good_results = nutrient_results.get_valid_data().get_sample_data().dropna(subset='Nitrate mean')

rme_cleaned.loc['03-15-25':'04-01-25'].plot(y= ['two_wavelength_no3_mgl_correct'], ax=ax, rot=45)
plot_nutrient_data(good_results[good_results['AA500 Run Date'] == '06/11/2025 12:28:19'], '03/15/25', '04/01/25', 'RME', 'Nitrate', ax)


### Dobson

In [ ]:

fig, ax= plt.subplots(figsize=(11,8.5), nrows=2)


dobson_cleaned.loc['11-1-24':'12-10-24'].plot(y= ['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax[0], rot=45)
plot_nutrient_data(nutrient_results.get_sample_data().dropna(subset='Nitrate mean'), '11/1/24', '12/10/24', 'Dobson', 'Nitrate', ax[0])


dobson_cleaned.loc['1-1-25':'04-01-25'].plot(y= ['two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax[1], rot=45)
plot_nutrient_data(nutrient_results.get_sample_data().dropna(subset='Nitrate mean'), '1/1/25', '07/07/25', 'Dobson', 'Nitrate', ax[1])
fig.tight_layout()

In [ ]:

fig, ax= plt.subplots(figsize=(11,8.5))



dobson_cleaned.loc['03-23-25':'03-27-25'].plot(y= ['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl', 'two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], ax=ax, rot=45)
plot_nutrient_data(nutrient_results.get_sample_data().dropna(subset='Nitrate mean'), '3/23/25', '03/27/25', 'Dobson', 'Nitrate', ax)


- not a lot of data points with S:CAN AND nutrient analyzer for Dobson yet

## Correlation Plots

### RME

In [ ]:
rme_aa500 =  nutrient_results.get_sample_data().dropna(subset='Nitrate mean').loc['RME']

rme_merged = pd.merge_asof(rme_aa500, rme_cleaned, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h'))


In [ ]:
rme_merged_clean = rme_merged.drop(['2/15/25 3:30:00', '2-22-25 11:00']).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct']) # drop two outliers from timeseries

In [ ]:
rme_merged_clean['two_wavelength_no3_mgl_correct'].isna().sum()

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(11,8.5))

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .2, .02, ax[0,0])

rme_merged_clean.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Corrected One Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, 2, .3, ax[1,0])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Uncorrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[0,1])

rme_merged_clean.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .03, ax[1,1])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,2], title='Uncorrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[0,2])

rme_merged_clean.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,2], title='Corrected Second Derivative')
plot_linear_fit(rme_merged_clean, rme_merged_clean, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[1,2])


fig.suptitle('RME Calibration Plots')
fig.tight_layout()


### Dobson

In [ ]:
dobson_aa500 =  nutrient_results.get_valid_data().get_sample_data().dropna(subset='Nitrate mean').loc['Dobson']

dobson_merged = pd.merge_asof(dobson_aa500, dobson_cleaned, left_index=True, right_index=True, direction='nearest', tolerance = pd.Timedelta('1h')).dropna(subset=['Nitrate mean', 'two_wavelength_no3_mgl_correct'])


In [ ]:
dobson_merged[dobson_merged['two_wavelength_no3_mgl_correct']>.6]

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(11,8.5))

dobson_merged.plot(x='Nitrate mean', y='one_wavelength_no3_mgl', kind='scatter', xerr='Nitrate err', ax=ax[0,0], title='Uncorrected One Wavelength')
plot_linear_fit(dobson_merged, dobson_merged, 'Nitrate mean', 'one_wavelength_no3_mgl', .3, .2, .02, ax[0,0])

dobson_merged.plot(x='Nitrate mean', y='one_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,0], title='Corrected One Wavelength')
plot_linear_fit(dobson_merged, dobson_merged, 'Nitrate mean', 'one_wavelength_no3_mgl_correct', .3, 2, .3, ax[1,0])

dobson_merged.plot(x='Nitrate mean', y='two_wavelength_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,1], title='Uncorrected Two Wavelength')
plot_linear_fit(dobson_merged, dobson_merged, 'Nitrate mean', 'two_wavelength_no3_mgl', .3, .15, .02, ax[0,1])

dobson_merged.plot(x='Nitrate mean', y='two_wavelength_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,1], title='Corrected Two Wavelength')
plot_linear_fit(dobson_merged, dobson_merged, 'Nitrate mean', 'two_wavelength_no3_mgl_correct', .3, .2, .03, ax[1,1])

dobson_merged.plot(x='Nitrate mean', y='second_derivative_no3_mgl', xerr='Nitrate err',kind='scatter', ax=ax[0,2], title='Uncorrected Second Derivative')
plot_linear_fit(dobson_merged, dobson_merged, 'Nitrate mean', 'second_derivative_no3_mgl', .3, .3, .03, ax[0,2])

dobson_merged.plot(x='Nitrate mean', y='second_derivative_no3_mgl_correct', xerr='Nitrate err',kind='scatter', ax=ax[1,2], title='Corrected Second Derivative')
plot_linear_fit(dobson_merged, dobson_merged, 'Nitrate mean', 'second_derivative_no3_mgl_correct', .3, .3, .03, ax[1,2])


fig.suptitle('Dobson Calibration Plots')
fig.tight_layout()
